# NB06 - Data Augmentation for Robustness
## Background
Data augmentation creates additional training examples by transforming existing ones, reducing overfitting and improving robustness to distribution shift. Classical augmentation (crops, flips, color jitter) was hand-crafted. AutoAugment (Cubuk et al., 2019) searched for optimal augmentation policies using reinforcement learning. RandAugment (Cubuk et al., 2020) simplified this with just two hyperparameters: magnitude and number of transforms per image.

AugMix (Hendrycks et al., 2020) improves robustness to distribution shifts (corruptions like blur, noise, weather effects) by mixing augmented chains and enforcing consistency via Jensen-Shannon divergence regularization. For tabular data, SMOTE, mixup, and noise injection serve similar roles.

## What You'll Learn
- Building an augmentation policy pipeline with composition and magnitude
- Augmentation search: evaluating policies on validation loss
- Mixup training: interpolating between examples and labels
- Robustness evaluation: testing under distribution shift

## Why This Matters
Production ML systems face distribution shift continuously. Augmentation designed for the expected shift patterns (sensor noise, lighting variation, domain adaptation) is often more effective than collecting more data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Callable

# ── Tabular augmentation operations ───────────────────────────────────────
def add_gaussian_noise(X: np.ndarray, magnitude: float = 0.1) -> np.ndarray:
    return X + np.random.randn(*X.shape) * magnitude

def feature_dropout(X: np.ndarray, magnitude: float = 0.1) -> np.ndarray:
    """Zero out random features with probability = magnitude."""
    mask = np.random.rand(*X.shape) > magnitude
    return X * mask

def feature_scaling(X: np.ndarray, magnitude: float = 0.2) -> np.ndarray:
    """Random per-feature scaling."""
    scale = 1.0 + np.random.randn(*X.shape) * magnitude
    return X * scale

def mixup(X1: np.ndarray, y1: np.ndarray, X2: np.ndarray, y2: np.ndarray,
          alpha: float = 0.4) -> Tuple[np.ndarray, np.ndarray]:
    """Mixup: linear interpolation of inputs and one-hot labels."""
    lam = np.random.beta(alpha, alpha)
    X_mix = lam * X1 + (1 - lam) * X2
    # Soft labels
    n_classes = max(y1.max(), y2.max()) + 1
    y1_oh = np.eye(n_classes)[y1]
    y2_oh = np.eye(n_classes)[y2]
    y_mix = lam * y1_oh + (1 - lam) * y2_oh
    return X_mix, y_mix

# ── Augmentation pipeline ──────────────────────────────────────────────────
class AugmentationPolicy:
    def __init__(self, ops: List[Tuple[Callable, float]], n_ops: int = 2):
        self.ops = ops  # List of (transform_fn, magnitude)
        self.n_ops = n_ops

    def apply(self, X: np.ndarray) -> np.ndarray:
        X = X.copy()
        selected = np.random.choice(len(self.ops), self.n_ops, replace=False)
        for idx in selected:
            fn, mag = self.ops[idx]
            X = fn(X, mag)
        return X

    @classmethod
    def identity(cls):
        return cls([])

    def apply_identity(self, X):
        return X

# ── Dataset generation ─────────────────────────────────────────────────────
np.random.seed(42)
n_train, n_val, n_test_clean, n_test_corrupt = 400, 100, 200, 200
n_classes, n_features = 4, 8

class_means = np.random.randn(n_classes, n_features) * 2
def gen_data(n, spread=1.0, seed=0):
    np.random.seed(seed)
    X, y = [], []
    for c in range(n_classes):
        X.append(class_means[c] + np.random.randn(n//n_classes, n_features)*spread)
        y.extend([c]*(n//n_classes))
    return np.vstack(X), np.array(y)

X_train, y_train = gen_data(n_train, spread=1.0, seed=1)
X_val, y_val     = gen_data(n_val,   spread=1.0, seed=2)
X_clean, y_clean = gen_data(n_test_clean, spread=1.0, seed=3)
X_corrupt, y_corrupt = gen_data(n_test_corrupt, spread=3.0, seed=4)  # High variance shift

print('Training data ready.')
print(f'Clean test set mean std: {X_clean.std():.3f}')
print(f'Corrupt test set mean std: {X_corrupt.std():.3f}')

In [ ]:
# ── Compare augmentation policies ─────────────────────────────────────────
class SimpleClf:
    def __init__(self, seed=0):
        np.random.seed(seed)
        self.W = np.random.randn(n_features, n_classes) * 0.1
        self.b = np.zeros(n_classes)

    def softmax(self, X):
        z = X @ self.W + self.b
        z -= z.max(axis=1, keepdims=True)
        e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

    def update(self, X, y, lr=0.1):
        if y.ndim == 2:  # Soft labels (mixup)
            probs = self.softmax(X)
            n = len(X)
            dz = (probs - y) / n
        else:
            probs = self.softmax(X)
            n = len(X)
            dz = probs.copy(); dz[np.arange(n), y] -= 1; dz /= n
        self.W -= lr * X.T @ dz
        self.b -= lr * dz.sum(axis=0)

    def accuracy(self, X, y):
        return np.mean(np.argmax(self.softmax(X), axis=1) == y)

ops = [
    (add_gaussian_noise, 0.3),
    (feature_dropout, 0.2),
    (feature_scaling, 0.3),
]

policies = {
    'No augmentation':  None,
    'Noise only':       AugmentationPolicy([(add_gaussian_noise, 0.3)], n_ops=1),
    'RandAugment-like': AugmentationPolicy(ops, n_ops=2),
    'Mixup':            'mixup',
}

results = {}
n_epochs = 80
batch_size = 32

for name, policy in policies.items():
    clf = SimpleClf(seed=42)
    for epoch in range(n_epochs):
        perm = np.random.permutation(len(X_train))
        for i in range(0, len(X_train), batch_size):
            idx = perm[i:i+batch_size]
            X_batch, y_batch = X_train[idx], y_train[idx]
            if policy is None:
                clf.update(X_batch, y_batch, lr=0.1)
            elif policy == 'mixup':
                idx2 = np.random.choice(len(X_train), len(idx))
                X_mix, y_mix = mixup(X_batch, y_batch, X_train[idx2], y_train[idx2])
                clf.update(X_mix, y_mix, lr=0.1)
            else:
                X_aug = policy.apply(X_batch)
                clf.update(X_aug, y_batch, lr=0.1)

    acc_clean = clf.accuracy(X_clean, y_clean)
    acc_corrupt = clf.accuracy(X_corrupt, y_corrupt)
    results[name] = {'clean': acc_clean, 'corrupt': acc_corrupt}
    print(f'{name:22s}: clean={acc_clean:.3f}, corrupt={acc_corrupt:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
names = list(results.keys())
clean_accs = [results[n]['clean'] for n in names]
corrupt_accs = [results[n]['corrupt'] for n in names]

x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, clean_accs, width, label='Clean test set', color='steelblue', alpha=0.8)
ax.bar(x + width/2, corrupt_accs, width, label='Corrupted test set', color='coral', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_title('Augmentation vs Robustness to Distribution Shift')
ax.set_ylabel('Test accuracy'); ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{BASE}/s29_06_augmentation.png', dpi=100, bbox_inches='tight')
plt.show()
print('Key insight: augmentation that matches the test-time distribution shift helps most.')